# 06. Temporal Partitioning

## Objectives

- Define reproducible temporal partitions.
- Preserve the experimental boundaries from EDA.
- Generate transaction-level temporal indices.
- Generate label-independent block manifests.
- Separate policy tuning from final evaluation.
- Prepare temporal metadata for label-delay simulation.

### Environment Setup

In [2]:
from pathlib import Path
import sys
import gc
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining


### Import Modules

In [3]:
from src.data_loader import (
    load_config,
    resolve_path,
)

from src.preprocessing import (
    save_parquet,
)

from src.data_profiling import (
    export_report,
)

from src.temporal_split import (
    build_ieee_temporal_index,
    build_baf_temporal_index,
    attach_temporal_index,
    build_phase_summary,
    build_block_statistics,
    validate_temporal_index,
)

config = load_config()

temporal_dir = resolve_path(config["paths"]["temporal_dir"])

report_dir = resolve_path(config["paths"]["temporal_report_dir"])

temporal_dir.mkdir(
    parents=True,
    exist_ok=True,
)

report_dir.mkdir(
    parents=True,
    exist_ok=True,
)

validation_records = []

print("Configuration loaded successfully.")

Configuration loaded successfully.


## 2. IEEE-CIS Temporal Partitioning

The original EDA boundaries are preserved.

- Warmup: days 1–30.
- Validation: days 31–86.
- Test: days 87–177.
- Tail: days 178–183.

Evaluation blocks contain seven consecutive days.

Temporal indices are stored separately from
processed feature datasets.

### Load Processed Metadata

In [4]:
ieee_config = config["datasets"]["ieee_cis"]

metadata_columns = [
    "TransactionID",
    "TransactionDT",
    "isFraud",
]

ieee_metadata = pd.read_parquet(
    resolve_path(ieee_config["processed_nocardid_path"]),
    columns=metadata_columns,
)

full_metadata = pd.read_parquet(
    resolve_path(ieee_config["processed_full_path"]),
    columns=metadata_columns,
)

pd.testing.assert_frame_equal(
    ieee_metadata,
    full_metadata,
)

del full_metadata

gc.collect()

print("Metadata shape:", ieee_metadata.shape)

display(ieee_metadata.head())

Metadata shape: (590540, 3)


,TransactionID,TransactionDT,isFraud
0,2987000,86400,0
1,2987001,86401,0
2,2987002,86469,0
3,2987003,86499,0
4,2987004,86506,0


### Generate Temporal Index

In [5]:
(
    ieee_temporal_index,
    ieee_manifest,
    time_origin,
) = build_ieee_temporal_index(ieee_metadata)

print("Time origin:", time_origin)

print(
    "Temporal index shape:",
    ieee_temporal_index.shape,
)

print(
    "First day:",
    ieee_temporal_index["day"].min(),
)

print(
    "Last day:",
    ieee_temporal_index["day"].max(),
)

display(ieee_temporal_index.head())

display(ieee_temporal_index.tail())

Time origin: 86400
Temporal index shape: (590540, 5)
First day: 1
Last day: 182


,TransactionID,TransactionDT,day,phase,block_id
0,2987000,86400,1,warmup,0
1,2987001,86401,1,warmup,0
2,2987002,86469,1,warmup,0
3,2987003,86499,1,warmup,0
4,2987004,86506,1,warmup,0


,TransactionID,TransactionDT,day,phase,block_id
590535,3577535,15811047,182,tail,-1
590536,3577536,15811049,182,tail,-1
590537,3577537,15811079,182,tail,-1
590538,3577538,15811088,182,tail,-1
590539,3577539,15811131,182,tail,-1


### Inspect Block Manifest

In [6]:
display(ieee_manifest)

print(
    "Evaluation blocks:",
    len(ieee_manifest),
)

print(ieee_manifest["phase"].value_counts())

,block_id,phase,start_day,end_day,start_time,end_time_exclusive,duration_days
0,1,validation,31,37,2678400,3283200,7
1,2,validation,38,44,3283200,3888000,7
2,3,validation,45,51,3888000,4492800,7
3,4,validation,52,58,4492800,5097600,7
4,5,validation,59,65,5097600,5702400,7
5,6,validation,66,72,5702400,6307200,7
6,7,validation,73,79,6307200,6912000,7
7,8,validation,80,86,6912000,7516800,7
8,9,test,87,93,7516800,8121600,7
9,10,test,94,100,8121600,8726400,7


Evaluation blocks: 21
phase
test          13
validation     8
Name: count, dtype: int64


### Validate Temporal Index

In [7]:
assert validate_temporal_index(
    temporal_index=ieee_temporal_index,
    manifest=ieee_manifest,
    id_column="TransactionID",
    time_column="TransactionDT",
    period_column="day",
    expected_rows=590540,
)

assert set(ieee_temporal_index.columns) == {
    "TransactionID",
    "TransactionDT",
    "day",
    "phase",
    "block_id",
}

assert set(ieee_temporal_index["phase"].unique()) == {
    "warmup",
    "validation",
    "test",
    "tail",
}

print("IEEE-CIS temporal validation passed.")

IEEE-CIS temporal validation passed.


### Attach Ground Truth for Offline Statistics

In [8]:
ieee_audit_df = attach_temporal_index(
    df=ieee_metadata,
    temporal_index=ieee_temporal_index,
    id_column="TransactionID",
    time_column="TransactionDT",
)

assert len(ieee_audit_df) == 590540

assert int(ieee_audit_df["isFraud"].sum()) == 20663

display(ieee_audit_df.head())

,TransactionID,TransactionDT,isFraud,day,phase,block_id
0,2987000,86400,0,1,warmup,0
1,2987001,86401,0,1,warmup,0
2,2987002,86469,0,1,warmup,0
3,2987003,86499,0,1,warmup,0
4,2987004,86506,0,1,warmup,0


### Phase Summary

In [9]:
ieee_phase_summary = build_phase_summary(
    ieee_audit_df,
    target_column="isFraud",
    temporal_column="day",
)

display(ieee_phase_summary)

,phase,start_period,end_period,observed_periods,num_transactions,num_fraud,num_legitimate,fraud_rate_pct
0,warmup,1,30,30,134339,3401,130938,2.531655
1,validation,31,86,56,169439,6881,162558,4.061049
2,test,87,177,91,274406,9888,264518,3.603420
3,tail,178,182,5,12356,493,11863,3.989964


### Block Statistics

In [10]:
ieee_block_statistics = build_block_statistics(
    ieee_audit_df,
    target_column="isFraud",
    temporal_column="day",
).rename(
    columns={
        "start_period": "start_day",
        "end_period": "end_day",
        "observed_periods": "observed_days",
    }
)

display(ieee_block_statistics)

assert len(ieee_block_statistics) == 21

assert (ieee_block_statistics["observed_days"] == 7).all()

assert (ieee_block_statistics["num_fraud"] > 0).all()

assert (ieee_block_statistics["num_legitimate"] > 0).all()

assert not ieee_block_statistics["low_fraud_warning"].any()

print("All evaluation blocks are valid.")

,block_id,phase,start_day,end_day,observed_days,num_transactions,num_fraud,num_legitimate,fraud_rate_pct,low_fraud_warning
0,1,validation,31,37,7,23467,863,22604,3.677505,False
1,2,validation,38,44,7,20710,821,19889,3.964268,False
2,3,validation,45,51,7,20334,867,19467,4.263795,False
3,4,validation,52,58,7,19554,750,18804,3.835532,False
4,5,validation,59,65,7,23003,1133,21870,4.925445,False
5,6,validation,66,72,7,21019,825,20194,3.925020,False
6,7,validation,73,79,7,20796,847,19949,4.072899,False
7,8,validation,80,86,7,20556,775,19781,3.770189,False
8,9,test,87,93,7,26932,898,26034,3.334323,False
9,10,test,94,100,7,23006,791,22215,3.438234,False


All evaluation blocks are valid.


### Verify Transaction Coverage

In [11]:
assert ieee_phase_summary["num_transactions"].sum() == len(ieee_metadata)

assert ieee_phase_summary["num_fraud"].sum() == ieee_metadata["isFraud"].sum()

evaluation_count = int(ieee_block_statistics["num_transactions"].sum())

evaluation_fraud = int(ieee_block_statistics["num_fraud"].sum())

assert evaluation_count == 443845

assert evaluation_fraud == 16769

print("Evaluation transactions:", evaluation_count)

print("Evaluation fraud:", evaluation_fraud)

print("Full dataset coverage verified.")

Evaluation transactions: 443845
Evaluation fraud: 16769
Full dataset coverage verified.


### Save IEEE-CIS Temporal Index

In [12]:
ieee_index_path = save_parquet(
    ieee_temporal_index,
    "data/temporal/ieee_temporal_index.parquet",
)

validation_records.append(
    {
        "dataset": "IEEE-CIS",
        "rows": len(ieee_temporal_index),
        "evaluation_blocks": len(ieee_manifest),
        "validation_blocks": 8,
        "test_blocks": 13,
        "status": "passed",
    }
)

Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\temporal\ieee_temporal_index.parquet


## 3. Comparison with Previous EDA

Temporal partitions must reproduce the
transaction and fraud counts reported in
the exploratory data analysis.

Any discrepancy indicates a potential
temporal boundary or preprocessing issue.

### Load EDA Reference

In [13]:
eda_path = resolve_path(
    Path(config["paths"]["profiling_dir"]) / "ieee_cis" / "evaluation_block_statistics.csv"
)

eda_block_statistics = pd.read_csv(eda_path)

display(eda_block_statistics.head())

,block_id,phase,start_day,end_day,observed_days,num_transactions,num_fraud,num_legitimate,fraud_rate_pct,low_fraud_warning
0,1,validation,31,37,7,23467,863,22604,3.677505,False
1,2,validation,38,44,7,20710,821,19889,3.964268,False
2,3,validation,45,51,7,20334,867,19467,4.263795,False
3,4,validation,52,58,7,19554,750,18804,3.835532,False
4,5,validation,59,65,7,23003,1133,21870,4.925445,False


### Compare All Blocks

In [14]:
comparison_columns = [
    "block_id",
    "phase",
    "start_day",
    "end_day",
    "observed_days",
    "num_transactions",
    "num_fraud",
    "num_legitimate",
]

old_stats = (
    eda_block_statistics[comparison_columns]
    .sort_values("block_id")
    .reset_index(drop=True)
)

new_stats = (
    ieee_block_statistics[comparison_columns]
    .sort_values("block_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    old_stats,
    new_stats,
    check_dtype=False,
)

np.testing.assert_allclose(
    eda_block_statistics.sort_values("block_id")["fraud_rate_pct"].to_numpy(),
    ieee_block_statistics["fraud_rate_pct"].to_numpy(),
    rtol=1e-10,
    atol=1e-10,
)

print("All 21 blocks match previous EDA.")

All 21 blocks match previous EDA.


### Export Comparison Report

In [15]:
ieee_eda_comparison = old_stats.merge(
    new_stats,
    on=["block_id", "phase"],
    suffixes=("_eda", "_current"),
    validate="one_to_one",
)

ieee_eda_comparison["transaction_difference"] = (
    ieee_eda_comparison["num_transactions_current"]
    - ieee_eda_comparison["num_transactions_eda"]
)

ieee_eda_comparison["fraud_difference"] = (
    ieee_eda_comparison["num_fraud_current"] - ieee_eda_comparison["num_fraud_eda"]
)

ieee_eda_comparison["matched"] = ieee_eda_comparison["transaction_difference"].eq(
    0
) & ieee_eda_comparison["fraud_difference"].eq(0)

assert ieee_eda_comparison["matched"].all()

display(
    ieee_eda_comparison[
        [
            "block_id",
            "phase",
            "transaction_difference",
            "fraud_difference",
            "matched",
        ]
    ]
)

export_report(
    ieee_eda_comparison,
    report_dir / "ieee_eda_comparison.csv",
)

,block_id,phase,transaction_difference,fraud_difference,matched
0,1,validation,0,0,True
1,2,validation,0,0,True
2,3,validation,0,0,True
3,4,validation,0,0,True
4,5,validation,0,0,True
5,6,validation,0,0,True
6,7,validation,0,0,True
7,8,validation,0,0,True
8,9,test,0,0,True
9,10,test,0,0,True


Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\ieee_eda_comparison.csv


## 4. BAF Temporal Partitioning

BAF variants are independent temporal streams.

The original monthly resolution is preserved.

- Warmup: months 0–1.
- Validation: months 2–3.
- Test: months 4–7.

Each evaluation block corresponds to one month.

No artificial within-month ordering is introduced.

### Initialize BAF Reports

In [16]:
baf_config = config["datasets"]["baf"]

baf_phase_reports = []

baf_block_reports = []

baf_manifest_reports = []

### Process All BAF Variants

In [17]:
for variant in baf_config["variants"]:

    print("=" * 60)

    print("Processing:", variant)

    variant_slug = Path(variant).stem.lower().replace(" ", "_")

    processed_path = resolve_path(
        Path(baf_config["processed_directory"]) / f"{variant_slug}.parquet"
    )

    # 1. Load metadata only.
    baf_metadata = pd.read_parquet(
        processed_path,
        columns=[
            "record_id",
            "month",
            "fraud_bool",
        ],
    )

    # 2. Build temporal index.
    (
        baf_index,
        baf_manifest,
    ) = build_baf_temporal_index(baf_metadata)

    # 3. Validate temporal metadata.
    assert validate_temporal_index(
        temporal_index=baf_index,
        manifest=baf_manifest,
        id_column="record_id",
        time_column="month",
        period_column="month",
        expected_rows=1000000,
    )

    # 4. Join ground truth for offline audit.
    baf_audit_df = attach_temporal_index(
        df=baf_metadata,
        temporal_index=baf_index,
        id_column="record_id",
        time_column="month",
    )

    assert int(baf_audit_df["fraud_bool"].sum()) == int(
        baf_metadata["fraud_bool"].sum()
    )

    # 5. Build phase summary.
    phase_summary = build_phase_summary(
        baf_audit_df,
        target_column="fraud_bool",
        temporal_column="month",
    )

    phase_summary.insert(
        0,
        "variant",
        variant,
    )

    baf_phase_reports.append(phase_summary)

    # 6. Build block statistics.
    block_statistics = build_block_statistics(
        baf_audit_df,
        target_column="fraud_bool",
        temporal_column="month",
    ).rename(
        columns={
            "start_period": "start_month",
            "end_period": "end_month",
            "observed_periods": "observed_months",
        }
    )

    block_statistics.insert(
        0,
        "variant",
        variant,
    )

    assert len(block_statistics) == 6

    assert (block_statistics["observed_months"] == 1).all()

    assert (block_statistics["num_fraud"] > 0).all()

    assert (block_statistics["num_legitimate"] > 0).all()

    baf_block_reports.append(block_statistics)

    # 7. Collect block manifest.
    variant_manifest = baf_manifest.copy()

    variant_manifest.insert(
        0,
        "variant",
        variant,
    )

    baf_manifest_reports.append(variant_manifest)

    # 8. Save temporal index.
    save_parquet(
        baf_index,
        (f"data/temporal/baf/" f"{variant_slug}_temporal_index.parquet"),
    )

    # 9. Collect validation results.
    validation_records.append(
        {
            "dataset": f"BAF {variant}",
            "rows": len(baf_index),
            "evaluation_blocks": len(baf_manifest),
            "validation_blocks": 2,
            "test_blocks": 4,
            "status": "passed",
        }
    )

    print("Rows:", len(baf_index))

    print(
        "Evaluation blocks:",
        len(baf_manifest),
    )

    # 10. Release memory.
    del baf_metadata
    del baf_index
    del baf_audit_df

    gc.collect()

print("All BAF temporal indices generated.")

Processing: Base.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\temporal\baf\base_temporal_index.parquet
Rows: 1000000
Evaluation blocks: 6
Processing: Variant I.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\temporal\baf\variant_i_temporal_index.parquet
Rows: 1000000
Evaluation blocks: 6
Processing: Variant II.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\temporal\baf\variant_ii_temporal_index.parquet
Rows: 1000000
Evaluation blocks: 6
Processing: Variant III.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\temporal\baf\variant_iii_temporal_index.parquet
Rows: 1000000
Evaluation blocks: 6
Processing: Variant IV.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\temporal\baf\variant_iv_temporal_index.parquet
Rows: 1000000
Evaluation blocks: 6
Processing: Variant V.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\temporal\

### Combine BAF Reports

In [18]:
baf_phase_summary = pd.concat(
    baf_phase_reports,
    ignore_index=True,
)

baf_block_statistics = pd.concat(
    baf_block_reports,
    ignore_index=True,
)

baf_block_manifest = pd.concat(
    baf_manifest_reports,
    ignore_index=True,
)

display(baf_phase_summary)

display(baf_block_statistics)

display(baf_block_manifest)

,variant,phase,start_period,end_period,observed_periods,num_transactions,num_fraud,num_legitimate,fraud_rate_pct
0,Base.csv,warmup,0,1,2,260060,2698,257362,1.037453
1,Base.csv,validation,2,3,2,287915,2590,285325,0.899571
2,Base.csv,test,4,7,4,452025,5741,446284,1.270062
3,Variant I.csv,warmup,0,1,2,260060,2698,257362,1.037453
4,Variant I.csv,validation,2,3,2,287917,2590,285327,0.899565
5,Variant I.csv,test,4,7,4,452023,5741,446282,1.270068
6,Variant II.csv,warmup,0,1,2,260060,2698,257362,1.037453
7,Variant II.csv,validation,2,3,2,287915,2590,285325,0.899571
8,Variant II.csv,test,4,7,4,452025,5741,446284,1.270062
9,Variant III.csv,warmup,0,1,2,185304,1874,183430,1.011311


,variant,block_id,phase,start_month,end_month,observed_months,num_transactions,num_fraud,num_legitimate,fraud_rate_pct,low_fraud_warning
0,Base.csv,1,validation,2,2,1,136979,1198,135781,0.874587,False
1,Base.csv,2,validation,3,3,1,150936,1392,149544,0.922245,False
2,Base.csv,3,test,4,4,1,127691,1452,126239,1.137120,False
3,Base.csv,4,test,5,5,1,119323,1411,117912,1.182505,False
4,Base.csv,5,test,6,6,1,108168,1450,106718,1.340507,False
5,Base.csv,6,test,7,7,1,96843,1428,95415,1.474552,False
6,Variant I.csv,1,validation,2,2,1,136979,1198,135781,0.874587,False
7,Variant I.csv,2,validation,3,3,1,150938,1392,149546,0.922233,False
8,Variant I.csv,3,test,4,4,1,127691,1452,126239,1.137120,False
9,Variant I.csv,4,test,5,5,1,119322,1411,117911,1.182515,False


,variant,block_id,phase,start_month,end_month,start_time,end_time_exclusive,duration_months
0,Base.csv,1,validation,2,2,2,3,1
1,Base.csv,2,validation,3,3,3,4,1
2,Base.csv,3,test,4,4,4,5,1
3,Base.csv,4,test,5,5,5,6,1
4,Base.csv,5,test,6,6,6,7,1
5,Base.csv,6,test,7,7,7,8,1
6,Variant I.csv,1,validation,2,2,2,3,1
7,Variant I.csv,2,validation,3,3,3,4,1
8,Variant I.csv,3,test,4,4,4,5,1
9,Variant I.csv,4,test,5,5,5,6,1


### Validate BAF Coverage

In [19]:
assert len(baf_phase_summary) == 18

assert len(baf_block_statistics) == 36

assert len(baf_block_manifest) == 36

assert not baf_block_manifest.duplicated(["variant", "block_id"]).any()

variant_counts = baf_phase_summary.groupby("variant")["num_transactions"].sum()

assert variant_counts.eq(1000000).all()

display(variant_counts)

print("All BAF coverage checks passed.")

variant
Base.csv           1000000
Variant I.csv      1000000
Variant II.csv     1000000
Variant III.csv    1000000
Variant IV.csv     1000000
Variant V.csv      1000000
Name: num_transactions, dtype: int64

All BAF coverage checks passed.


### Compare with BAF EDA

In [20]:
baf_eda = pd.read_csv(
    resolve_path(Path(config["paths"]["profiling_dir"]) / "baf" /"baf_monthly_statistics.csv")
)

baf_eda["variant"] = (
    baf_eda["variant"].str.replace(
        r"\.csv$",
        "",
        regex=True,
    )
    + ".csv"
)

baf_eda_eval = baf_eda.loc[baf_eda["month"].between(2, 7)].copy()

baf_comparison = baf_block_statistics.merge(
    baf_eda_eval,
    left_on=[
        "variant",
        "start_month",
    ],
    right_on=[
        "variant",
        "month",
    ],
    suffixes=("_current", "_eda"),
    validate="one_to_one",
)

assert len(baf_comparison) == 36

assert (
    baf_comparison["num_transactions_current"] == baf_comparison["num_transactions_eda"]
).all()

assert (baf_comparison["num_fraud_current"] == baf_comparison["num_fraud_eda"]).all()

print("All BAF evaluation blocks match EDA.")

All BAF evaluation blocks match EDA.


## 5. Export Temporal Reports

Temporal reports document experimental boundaries,
dataset coverage, evaluation block statistics
and validation results.

Block manifests are label-independent.

Ground-truth statistics are reserved for
offline auditing and evaluation.

### Export CSV Reports

In [21]:
reports = {
    "ieee_phase_summary.csv": (ieee_phase_summary),
    "ieee_block_manifest.csv": (ieee_manifest),
    "ieee_block_statistics.csv": (ieee_block_statistics),
    "baf_phase_summary.csv": (baf_phase_summary),
    "baf_block_manifest.csv": (baf_block_manifest),
    "baf_block_statistics.csv": (baf_block_statistics),
    "temporal_validation.csv": (pd.DataFrame(validation_records)),
}

for filename, report in reports.items():

    export_report(
        report,
        report_dir / filename,
    )

Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\ieee_phase_summary.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\ieee_block_manifest.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\ieee_block_statistics.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\baf_phase_summary.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\baf_block_manifest.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\baf_block_statistics.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\temporal_validation.csv


### Export Temporal Metadata

In [22]:
temporal_metadata = {
    "version": "1.0.0",
    "interval_convention": ("[start_time, end_time_exclusive)"),
    "ieee_cis": {
        "time_origin": int(time_origin),
        "time_unit": "seconds",
        "seconds_per_day": 86400,
        "day_index_base": 1,
        "day_formula": ("floor((TransactionDT - time_origin)" " / 86400) + 1"),
        "first_day": 1,
        "last_day": 183,
        "warmup_days": [1, 30],
        "validation_days": [31, 86],
        "last_day": int(ieee_temporal_index["day"].max()),
        "tail_days": [
            int(
                ieee_temporal_index.loc[
                    ieee_temporal_index["phase"].eq("tail"), "day"
                ].min()
            ),
            int(
                ieee_temporal_index.loc[
                    ieee_temporal_index["phase"].eq("tail"), "day"
                ].max()
            ),
        ],
        "block_size_days": 7,
        "validation_blocks": 8,
        "test_blocks": 13,
        "warmup_block_id": 0,
        "tail_block_id": -1,
        "include_tail_in_evaluation": False,
        "primary_feature_set": "NoCardID",
    },
    "baf": {
        "time_unit": "month",
        "first_month": 0,
        "last_month": 7,
        "warmup_months": [0, 1],
        "validation_months": [2, 3],
        "test_months": [4, 7],
        "block_size_months": 1,
        "validation_blocks": 2,
        "test_blocks": 4,
        "warmup_block_id": 0,
        "primary_feature_set": "CommonFeatures",
        "within_month_order_available": False,
    },
}

metadata_path = report_dir / "temporal_metadata.json"

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        temporal_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved:", metadata_path)

Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\temporal\temporal_metadata.json


## 6. Final Verification

Verify persisted temporal indices and confirm
that every processed dataset can be linked
to its temporal metadata.

The original processed feature files
remain unchanged.

### Verify Persisted IEEE-CIS Index

In [23]:
saved_ieee_index = pd.read_parquet(temporal_dir / "ieee_temporal_index.parquet")

pd.testing.assert_frame_equal(
    saved_ieee_index,
    ieee_temporal_index,
)

assert validate_temporal_index(
    temporal_index=saved_ieee_index,
    manifest=ieee_manifest,
    id_column="TransactionID",
    time_column="TransactionDT",
    period_column="day",
    expected_rows=590540,
)

print("Persisted IEEE-CIS index verified.")

Persisted IEEE-CIS index verified.


### Verify Persisted BAF Indices

In [24]:
for variant in baf_config["variants"]:

    variant_slug = Path(variant).stem.lower().replace(" ", "_")

    saved_index = pd.read_parquet(
        temporal_dir / "baf" / f"{variant_slug}_temporal_index.parquet"
    )

    variant_manifest = baf_block_manifest.loc[
        baf_block_manifest["variant"].eq(variant)
    ].drop(columns="variant")

    assert validate_temporal_index(
        temporal_index=saved_index,
        manifest=variant_manifest,
        id_column="record_id",
        time_column="month",
        period_column="month",
        expected_rows=1000000,
    )

    processed_metadata = pd.read_parquet(
        resolve_path(
            Path(baf_config["processed_directory"]) / f"{variant_slug}.parquet"
        ),
        columns=["record_id", "month"],
    )

    checked = attach_temporal_index(
        processed_metadata,
        saved_index,
        id_column="record_id",
        time_column="month",
    )

    assert len(checked) == 1000000

    print(variant, ": passed")

    del saved_index
    del processed_metadata
    del checked

    gc.collect()

Base.csv : passed
Variant I.csv : passed
Variant II.csv : passed
Variant III.csv : passed
Variant IV.csv : passed
Variant V.csv : passed


### Final Summary

In [25]:
temporal_validation = pd.DataFrame(validation_records)

display(temporal_validation)

assert len(temporal_validation) == 7

assert temporal_validation["status"].eq("passed").all()

assert temporal_validation["rows"].sum() == 6590540

assert temporal_validation["evaluation_blocks"].sum() == 57

print("=" * 60)

print("STEP 4: TEMPORAL PARTITIONING COMPLETED")

print("Temporal indices: 7")

print("IEEE-CIS evaluation blocks: 21")

print("BAF evaluation blocks: 36")

print("Total indexed records: 6590540")

print("Ready for Step 5: Label Delay Simulation")

print("=" * 60)

,dataset,rows,evaluation_blocks,validation_blocks,test_blocks,status
0,IEEE-CIS,590540,21,8,13,passed
1,BAF Base.csv,1000000,6,2,4,passed
2,BAF Variant I.csv,1000000,6,2,4,passed
3,BAF Variant II.csv,1000000,6,2,4,passed
4,BAF Variant III.csv,1000000,6,2,4,passed
5,BAF Variant IV.csv,1000000,6,2,4,passed
6,BAF Variant V.csv,1000000,6,2,4,passed


STEP 4: TEMPORAL PARTITIONING COMPLETED
Temporal indices: 7
IEEE-CIS evaluation blocks: 21
BAF evaluation blocks: 36
Total indexed records: 6590540
Ready for Step 5: Label Delay Simulation
